# 01 描述性统计与 EDA 可视化分析

**模块职责**：基于角色 A 交付的主数据集 `data/processed/master_data.csv` 开展描述性统计，挖掘数据分布特征，产出多维度对比图表。

**输入契约**：主数据集为「球员-赛季」粒度，原始 78 列；`viz.load_master()` 会再派生 10 列（9 个 `*_per_game` 场均列 + `primary_pos` 主位置列），读入后为 88 列。绘图逻辑全部封装在 `src/visualization.py`，本 Notebook 只负责调用与解读，便于角色 E 的 Streamlit 看板复用同一套函数。

**两个必须知道的坑**：

1. `master_data.csv` 带 UTF-8 BOM，必须用 `encoding="utf-8-sig"` 读取，否则首列名会变成 `\ufeffYear`；`viz.load_master()` 已内置处理。
2. 三份原始数据均不含合同信息，**主数据集没有薪资列**。因此本 Notebook 用 `BPM` 分档构造「价值梯队」作为薪资梯队的代理变量，相关结论需按此口径理解。

**分析路线**：数据概览 → 质量审计 → 描述性统计 → 分布形态 → 相关性 → 分组对比（位置/时代/价值梯队）→ 时代演化 → 结论与下游建议

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import pandas as pd

# 从任意工作目录启动都能定位到仓库根目录（notebooks/ 的上一级）
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "__init__.py").exists()), None)
if ROOT is None:
    raise RuntimeError("未找到仓库根目录：请在 notebooks/ 目录或项目根目录下运行本 Notebook")
sys.path.insert(0, str(ROOT))

from src import visualization as viz

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

df = viz.load_master()
print(f"主数据集: {df.shape[0]} 行 x {df.shape[1]} 列")
print(f"赛季范围: {df['Year'].min():.0f} - {df['Year'].max():.0f}")
print(f"球员人数: {df['Player'].nunique()}")

## 一、数据概览

先确认粒度是否真的是「球员-赛季」唯一，以及各年代覆盖情况。

In [ ]:
dup = df.duplicated(subset=["Player", "Year", "Age"]).sum()
print(f"Player+Year+Age 重复行数: {dup}")
print(f"平均每人覆盖赛季数: {len(df) / df['Player'].nunique():.2f}")
df[["Year", "Player", "Pos", "Age", "Tm", "G", "MP", "PTS", "PER", "TS%", "BPM", "era"]].head()

In [ ]:
print("缺失率最高的 15 列（%）")
(df.isna().mean() * 100).sort_values(ascending=False).head(15).round(2)

In [ ]:
print("各年代记录数、球员数与赛季总分均值")
df.groupby("era", observed=True).agg(
    记录数=("Player", "size"), 球员数=("Player", "nunique"), 赛季总分均值=("PTS", "mean")
).round(1)

## 二、数据质量审计（本节结论直接影响角色 C / D）

EDA 的首要产出不是图表，而是**这份数据能不能直接建模**。审计发现三个必须处理的问题。

### 问题 1：缺失是结构性的，与时代绑定，不能盲目 fillna(0)

`3PAr` 缺 23.7%（4813 行）、`3P%` 缺 36.1%（7341 行）—— NBA 1979-80 赛季才引入三分线，本数据中 `3PAr` 最早出现在 1980 赛季，1980 年及以后只缺 44 行；`3P%` 另有 2572 行是 1980 年后 `3PA == 0`（一次三分都没投，命中率无定义）。`BPM/VORP/OBPM/DBPM` 齐缺 15.8%（3214 行），全部落在 1974 赛季之前，1974 年起零缺失。这些「缺失」的含义是**当时根本没统计**或**分母为 0**，而不是「球员该能力为 0」。填 0 会凭空造出一批「零三分、零影响力」的假球员。

In [ ]:
print("缺失率是否由时代决定？按年代看各指标的缺失率（%）")
display(
    df.groupby("era", observed=True)[["3PAr", "3P%", "BPM", "USG%"]]
    .apply(lambda g: (g.isna().mean() * 100).round(1))
)

print("\n1980 年及以后的缺失是「没统计」还是「分母为 0」？")
modern = df[df["Year"] >= 1980]
zero_3pa = ((modern["3PA"] == 0) & modern["3P%"].isna()).sum()
print(f"1980 及以后共 {len(modern)} 行：3PAr 仍缺 {modern['3PAr'].isna().sum()} 行；"
      f"3P% 仍缺 {modern['3P%'].isna().sum()} 行，其中 {zero_3pa} 行是 3PA==0（一次三分都没投）")
print(f"BPM：1974 之前 {df.loc[df['Year'] < 1974, 'BPM'].isna().sum()} 行全缺，"
      f"1974 及以后缺 {df.loc[df['Year'] >= 1974, 'BPM'].isna().sum()} 行")


### 问题 2：极小样本行制造统计假象（最危险）

数据里存在大量 `G=1、MP 只有几分钟` 的短工合同行。它们的比率型指标会取到荒谬极值，例如 `USG%=100`、`PER=-90.6`。下面直接看这些极值行是谁。

In [ ]:
print("PER 最低 5 行 / 最高 3 行 —— 注意 G 与 MP 列")
cols = ["Year", "Player", "Tm", "G", "MP", "PTS", "PER", "TS%", "USG%"]
pd.concat([df.nsmallest(5, "PER")[cols], df.nlargest(3, "PER")[cols]])

In [ ]:
print("TS% 超出合理区间（>0.75）与 USG% 达 100 的行")
pd.concat([df[df["TS%"] > 0.75][cols].head(5), df[df["USG%"] >= 99][cols].head(3)])

这些行本身不是错误数据，而是**样本量太小导致比率失真**。处理方式不是删掉球员，而是给分析设定统一的出场门槛。`viz.filter_qualified()` 默认要求 `MP >= 500 且 G >= 5`，即约一个赛季轮换以上的出场量。

In [ ]:
qdf = viz.filter_qualified(df)
print(f"合格样本: {len(qdf)} 行 / {qdf['Player'].nunique()} 人（占全量 {len(qdf) / len(df) * 100:.1f}%）")
print("过滤前后比率型指标取值范围对比：")
check_cols = ["PER", "TS%", "USG%", "PTS_per36"]
pd.DataFrame({
    "全量最小值": [df[c].min() for c in check_cols],
    "全量最大值": [df[c].max() for c in check_cols],
    "合格样本最小值": [qdf[c].min() for c in check_cols],
    "合格样本最大值": [qdf[c].max() for c in check_cols],
}, index=check_cols).round(2)

**过滤后 `PER` 落在 3.0~31.8、`TS%` 落在 0.30~0.73、`USG%` 上限 41.7，全部回到篮球学上的合理区间**，证实这些极值确实由小样本造成。后续分布与相关性分析统一使用 `qdf`。

### 问题 3：位置标签自带时代污染

主位置 `primary_pos` 中，裸标签 `F` / `G`（不细分前后锋）几乎只出现在上古年代，与现代的 `PG/SG/SF/PF/C` 不是同一套标注体系。

In [ ]:
print("各主位置的样本量与年代分布")
qdf.groupby("primary_pos", observed=True).agg(
    样本数=("Player", "size"), 起始年=("Year", "min"), 年代中位数=("Year", "median"), 结束年=("Year", "max")
)

## 三、描述性统计汇总表

计算均值、中位数、标准差、四分位与偏度。**口径提醒**：主数据集中的 `MP/PTS/TRB` 等是**赛季总量**而非场均（已用 Harden 2010 赛季 `G=76 / PTS=753` → 场均 9.9 分验证），因此 `viz.add_derived_columns()` 批量派生了 `*_per_game` 列，本节分析以场均口径为准。

In [ ]:
summary = viz.describe_summary(qdf)
summary

In [ ]:
viz.plot_describe_heatmap(summary);


## 四、得分分布

三种得分口径对照：`PTS`（赛季总分）、`PTS_per_game`（场均）、`PTS_per36`（每 36 分钟折算，角色 A 派生）。三者差异即上场时长带来的偏差。

In [ ]:
print("三种得分口径对照")
qdf[["PTS", "PTS_per_game", "PTS_per36"]].describe(percentiles=[.25, .5, .75, .95]).round(2)

In [ ]:
viz.plot_score_distribution(qdf, metric="PTS_per_game");


In [ ]:
viz.plot_score_distribution(qdf, metric="PTS_per_game", hue="primary_pos");


## 五、相关性分析

核心考察 README 指定的「得分分布 × 出场时间 × 效率值 PER」三者关系。基础计数指标之间会因上场时间产生机械性共线，因此同时看一张全指标矩阵和一张纯高阶效率矩阵。

In [ ]:
viz.plot_corr_heatmap(qdf);


In [ ]:
advanced = ["PER", "TS%", "USG%", "3PAr", "AST%", "TRB%", "STL%", "BLK%", "TOV%", "WS/48", "BPM", "MP_per_game"]
viz.plot_corr_heatmap(qdf, columns=advanced);


In [ ]:
viz.plot_scatter_with_trend(qdf, x="MP_per_game", y="PER");


In [ ]:
print("出场时间 / 产量 / 效率三者的相关结构（合格样本）")
qdf[["MP_per_game", "PTS_per_game", "PER", "USG%", "TS%", "BPM"]].corr().round(3)

## 六、分组对比：位置 / 时代 / 价值梯队

`value_tier` 是 BPM 分档构造的**薪资梯队代理变量**（原始数据无合同信息），分档规则见 `viz.add_value_tier` 的文档。

In [ ]:
viz.plot_group_box(qdf, metric="PTS_per_game", group="primary_pos");


In [ ]:
viz.plot_group_violin(qdf, metric="PER", group="era");


In [ ]:
tiered = viz.add_value_tier(qdf)
print("价值梯队分布（BPM 代理，按 Top10% / 50% / 20% 分位切分）")
print(tiered["value_tier"].value_counts().sort_index())
print()
print("各梯队的场均数据中位数")
tiered.groupby("value_tier", observed=True)[["PTS_per_game", "MP_per_game", "PER", "TS%", "USG%", "BPM"]].median().round(2)

In [ ]:
viz.plot_group_box(tiered, metric="PTS_per36", group="value_tier");


In [ ]:
viz.plot_group_median_bar(qdf, metric="3PAr", group="era");


## 七、时代演化趋势

验证「无位置篮球 / 三分革命」是否在数据中成立——这是角色 C 做聚类前的重要前提。

In [ ]:
viz.plot_metric_trend(qdf, metric="3PAr");


In [ ]:
viz.plot_metric_trend(qdf, metric="AST%");


In [ ]:
print("按年代的打法特征中位数")
qdf.groupby("era", observed=True)[["3PAr", "TS%", "USG%", "AST%", "MP_per_game", "Age"]].median().round(3)

## 八、EDA 结论与下游建议

以下每条结论都对应上文某个单元格的实测输出，且**统一使用合格样本 `qdf` 口径**。

### 1. 数据边界：止于 2017 赛季

主数据集覆盖 **1950–2017**，共 20313 行、3921 名球员，2018 年及以后为 0 行。这直接约束角色 D：「预测下赛季 PER」最多只能构造到 2016→2017 的转移对；若课题要求覆盖近几个赛季，需要角色 A 更换或补采数据源，而不是在 B/C/D 层修补。

### 2. 统一分析口径：`viz.filter_qualified()`，14860 行 / 2508 人（占全量 73.2%）

小样本行使 `PER` 出现 -90.6、`TS%` 出现 1.136、`USG%` 出现 100 等无意义极值；过滤后三者分别回到 3.0~31.8、0.30~0.73、上限 41.7。**建议角色 C 的聚类与角色 D 的建模都在这份合格样本上进行**，且门槛口径要与 B 保持一致，否则三方结果无法互相印证。

### 3. 缺失是时代造成的，禁止 fillna(0)

`3PAr` 的缺失几乎全部落在 1979 赛季及更早（三分线未引入，1980 年及以后仅缺 44 行），`BPM`/`VORP` 全部落在 1974 赛季之前；`3P%` 另有 2572 行是 1980 年后 `3PA == 0`（出手为零，命中率无定义）。填 0 等于宣称上古球员「三分能力为零」、宣称纯两分球员「三分全部打铁」。更稳妥的做法是只取 1997 年及以后子样本做聚类，或把缺失本身编码成「该年代未统计」指示列。

### 4. 得分分布右偏，建模前建议做变换

合格样本 `PTS_per_game` 偏度 0.935，中位数 9.4、P95 22.0、最大值 50.4，是典型右偏长尾。K-Means 基于欧氏距离，长尾会拉扯质心，角色 C 建议先做 log1p 或分位数标准化；角色 D 若以得分为回归目标，同样应对数变换。

### 5. 效率与产量近似正交，是聚类的良好坐标轴

合格样本中 `TS%` 与 `USG%` 相关仅 0.124，说明「投得准」和「扛球权多」基本是两件事；而 `PER` 与 `BPM` 相关 0.768、`MP_per_game` 与 `PTS_per_game` 相关 0.863，属于高度冗余的同类信息。角色 C 选特征时应避免把 `PER/BPM/WS/48` 全部塞进模型（会重复计权），更适合用「产量轴 × 效率轴」的正交组合。

### 6. 三分革命确证，传统位置标签失效

合格样本上 `3PAr` 中位数从 1980s 的 0.014 升到 2010s 的 0.264（约 19 倍），同期 `AST%` 由 12.0 降至 10.6——打法明显从团队传导转向空间型单挑。同时裸位置标签 `F`/`G` 的年代中位数为 1955，而 `PG/SG/SF/PF/C` 为 1994~1996，说明 `Pos` 字段混用了两套不同时代的标注体系。**结论：角色 C 不应把 `Pos` 作为聚类特征**，只宜事后用于结果解读，这也正是本模块「打破 5 个标称位置」命题的实证依据。

### 7. 价值梯队代理可用，但需在报告中标注

BPM 分档得到的四个梯队在场均得分（5.6 → 7.9 → 11.5 → 18.2）、场均出场（16.4 → 21.5 → 29.0 → 35.4 分钟）与 `TS%`（0.48 → 0.52 → 0.54 → 0.57）上均单调递增，作为「薪资梯队」的替身在描述性分析中成立。但它是能力代理而非真实合同金额，最终报告与看板（角色 E）必须显式标注这一点，避免评委误读。

In [ ]:
# 一键复现给下游的口径：角色 C / D / E 直接沿用下面两行即可
from src import visualization as viz

qdf = viz.filter_qualified(viz.load_master())
print(f"合格样本 {qdf.shape[0]} 行 / {qdf['Player'].nunique()} 人，赛季 {qdf['Year'].min():.0f}-{qdf['Year'].max():.0f}")

cluster_features = ["MP_per_game", "PTS_per_game", "TRB_per_game", "AST_per_game",
                    "STL_per_game", "BLK_per_game", "TOV_per_game",
                    "3PAr", "FTr", "TS%", "eFG%", "USG%"]
present = [c for c in cluster_features if c in qdf.columns]
print(f"建议用于聚类的正交特征列（{len(present)} 个，均已在合格样本中）：")
print(present)